# 8. Revision — External (Table 3) Harmonization

**Purpose.** Report the external-validation results (Table 3) for the SAME finalized model that the manuscript reports internally and that is deployed in the calculator (`model/final_model_calibrated.pkl`, the Table S2 hyperparameters). All three comparators are trained on the derivation cohort and applied once to the external cohort; each AUC point estimate and its 95% bootstrap CI come from the SAME external predictions (correcting a prior point/CI mismatch for the AJCC row). The MSKCC comparator uses the adapted variable set (excluding postoperative chemotherapy), constructed exactly as in notebook 4.

Harness: `random_state = 8251`, KNNImputer(k=5), LogisticRegression(C=1e9, class_weight='balanced') for the AJCC/MSKCC comparators, the deployed calibrated XGBoost for the four-variable model.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, joblib
from pathlib import Path
import scipy.stats as st
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
SEED = 8251

def find_data_dir():
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base/'local_data', base/'stage_III_colon_edr'/'local_data'):
            if cand.exists(): return cand
    raise FileNotFoundError('Place a local_data/ folder (prepared parquet/CSV) in the repo root; see README.')

def find_model_dir():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base/'model'/'final_model_calibrated.pkl').exists(): return base/'model'
    raise FileNotFoundError('model/ not found')

DATA = find_data_dir(); MODEL = find_model_dir()
deriv = pd.read_parquet(DATA/'all_cases_prepared_for_ML.parquet')
ext   = pd.read_csv(DATA/'data_typed_ext.csv')
feat_cols = list(joblib.load(MODEL/'final_feature_columns.pkl'))
PKL = joblib.load(MODEL/'final_model_calibrated.pkl')   # the deployed Table S2 model
print('Derivation n=%d (events %d) | External n=%d (events %d)' % (len(deriv), int(deriv.edr_18m.sum()), len(ext), int(ext.edr_18m.astype(int).sum())))

Derivation n=331 (events 62) | External n=142 (events 19)


## Feature construction (identical to notebook 4)
MSKCC adapted variable set: age, tumor location, log(preoperative CEA), differentiation, lymphovascular invasion, perineural invasion, positive nodes, negative nodes, and T stage (postoperative chemotherapy omitted).

In [2]:
def pt_to_num(s):
    return pd.Series(s).astype(str).str.upper().str.replace('T','',regex=False).map({'1':1,'2':2,'3':3,'4A':4,'4B':5,'4':4})

def make(df):
    df = df.copy()
    Xxgb = pd.get_dummies(df[['AJCC_Substage','PNI','LNR','Differentiation']], columns=['AJCC_Substage'])
    Xxgb['PNI'] = Xxgb['PNI'].astype(float); Xxgb['Differentiation'] = Xxgb['Differentiation'].astype(float)
    Xxgb = Xxgb.replace([np.inf,-np.inf], np.nan)
    df['pT_num'] = pt_to_num(df['pT_Stage']); df['neg_nodes'] = df['LN_Total'] - df['LN_Positive']
    Xmskcc = df[['Age','Log_CEA_PreOp','Tumor_Location_Group','Differentiation','LVI','PNI','LN_Positive','neg_nodes','pT_num']].astype(float)
    Xajcc = Xxgb[[c for c in Xxgb.columns if 'AJCC' in c]].copy()
    return Xxgb, Xmskcc, Xajcc, df['edr_18m'].astype(int)

Xxgb_d, Xmskcc_d, Xajcc_d, y_d = make(deriv)
Xxgb_e, Xmskcc_e, Xajcc_e, y_e = make(ext)
Xxgb_e = Xxgb_e.reindex(columns=feat_cols, fill_value=0); y_e = y_e.reset_index(drop=True)

In [3]:
def boot_ci(y, p, n=1000, seed=SEED):
    rng = np.random.RandomState(seed); y = np.asarray(y); p = np.asarray(p); o = []
    for _ in range(n):
        i = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[i])) > 1: o.append(roc_auc_score(y[i], p[i]))
    return np.percentile(o, 2.5), np.percentile(o, 97.5)

def metrics_at(y, p, cut):
    yh = (np.asarray(p) >= cut).astype(int); tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0,1]).ravel()
    f = lambda a, b: a/b if b else np.nan
    return f(tp,tp+fn), f(tn,tn+fp), f(tp,tp+fp), f(tn,tn+fn)

def youden(y, p):
    fpr, tpr, th = roc_curve(y, p); return th[np.argmax(tpr - fpr)]

def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]: j += 1
        T[i:j] = 0.5*(i+j-1); i = j
    T2 = np.empty(N); T2[J] = T + 1; return T2

def delong(y, p1, p2):
    y = np.asarray(y); o = np.argsort(-y); y = y[o]; ps = np.vstack((np.asarray(p1)[o], np.asarray(p2)[o]))
    m = int(y.sum()); n = len(y) - m
    tx = np.empty((2,m)); ty = np.empty((2,n)); tz = np.empty((2,m+n))
    for r in range(2):
        tx[r] = _midrank(ps[r,:m]); ty[r] = _midrank(ps[r,m:]); tz[r] = _midrank(ps[r,:])
    aucs = tz[:,:m].sum(1)/m/n - (m+1)/2/n
    v01 = (tz[:,:m]-tx)/n; v10 = 1-(tz[:,m:]-ty)/m; S = np.cov(v01)/m + np.cov(v10)/n
    L = np.array([[1,-1]]); z = (aucs[0]-aucs[1]) / np.sqrt((L@S@L.T)[0,0] + 1e-12)
    return float(aucs[0]), float(aucs[1]), float(2*st.norm.sf(abs(z)))

def fit_pred(Xtr, ytr, Xte):
    lr = Pipeline([('imp', KNNImputer(n_neighbors=5)),
                   ('lr', LogisticRegression(C=1e9, class_weight='balanced', solver='liblinear', random_state=SEED))])
    lr.fit(Xtr, ytr); return lr.predict_proba(Xte)[:,1]

## External Table 3 (harmonized) and DeLong comparisons

In [4]:
p_xgb  = PKL.predict_proba(Xxgb_e)[:,1]
p_ajcc = fit_pred(Xajcc_d, y_d, Xajcc_e)
p_msk  = fit_pred(Xmskcc_d, y_d, Xmskcc_e)
rows = []
for name, p, cut in [('AJCC substage only', p_ajcc, youden(y_e, p_ajcc)),
                     ('Adapted MSKCC-variable model', p_msk, youden(y_e, p_msk)),
                     ('Four-variable XGBoost', p_xgb, 0.120)]:
    auc = roc_auc_score(y_e, p); lo, hi = boot_ci(y_e, p); se, sp, pv, nv = metrics_at(y_e.values, p, cut)
    rows.append({'Model': name, 'Cutoff': round(cut,3), 'AUC': round(auc,3), '95% CI': '%.3f-%.3f'%(lo,hi),
                 'Sens': '%.1f%%'%(se*100), 'Spec': '%.1f%%'%(sp*100), 'PPV': '%.1f%%'%(pv*100), 'NPV': '%.1f%%'%(nv*100)})
print(pd.DataFrame(rows).to_string(index=False))
print('\nDeLong (external):')
for bn, b in [('AJCC', p_ajcc), ('adapted MSKCC-variable model', p_msk)]:
    a1, a2, pp = delong(y_e.values, p_xgb, b)
    print('  XGBoost (%.3f) vs %s (%.3f):  p = %.3f' % (a1, bn, a2, pp))

                       Model  Cutoff   AUC      95% CI  Sens  Spec   PPV   NPV
          AJCC substage only   0.757 0.610 0.490-0.732 42.1% 76.4% 21.6% 89.5%
Adapted MSKCC-variable model   0.532 0.685 0.556-0.808 73.7% 62.6% 23.3% 93.9%
       Four-variable XGBoost   0.120 0.633 0.501-0.757 68.4% 48.0% 16.9% 90.8%

DeLong (external):
  XGBoost (0.633) vs AJCC (0.610):  p = 0.381
  XGBoost (0.633) vs adapted MSKCC-variable model (0.685):  p = 0.174


## Summary — Table 3 changes under harmonization

| Row | previously reported | harmonized (deployed model) |
|---|---|---|
| XGBoost AUC | 0.637 (0.503–0.761) | **0.633 (0.501–0.757)** |
| AJCC AUC | 0.617 (0.405–0.696) — point/CI mismatch | **0.610 (0.490–0.732)** |
| MSKCC AUC | 0.685 (0.556–0.808) | 0.685 (0.556–0.808) — unchanged |
| DeLong XGBoost vs AJCC | 0.291 | **0.381** |
| DeLong XGBoost vs MSKCC | 0.228 | **0.174** |

All threshold-based metrics (sensitivity, specificity, PPV, NPV; NPV 90.8% for the four-variable model) are unchanged. All comparisons remain non-significant. Internal results are reported in notebook 7.